# Run DETR model
* Use frames provided by Heba in the google drive.
* Data folder ~/scratch/2026_freeman_bird_behavior/data/wz_data (H23 or H03)
* The script of base DETR model was cloned from [DETR](https://github.com/facebookresearch/detr)

In [2]:
import sys
sys.path.append('/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior')

from detr_detector.models.detr import DETR
from detr_detector.models.backbone import build_backbone
from detr_detector.models.transformer import build_transformer
from detr_detector.engine import train_one_epoch, evaluate

# 1. Verify GPU

In [3]:
import torch
print(f"# GPU available:  {torch.cuda.is_available()}")
print(f"# GPU name:       {torch.cuda.get_device_name(0)}")
print(f"# VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# GPU available:  True
# GPU name:       NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 2g.48gb
# VRAM:           50.9 GB


# 2. Verify DETR model builds correctly:

In [4]:
import argparse
from detr_detector.models import build_model

device = 'cuda' if torch.cuda.is_available() else "cpu"
args = argparse.Namespace(num_classes=2, # 1 class: bird, but they mentioned this value is n_classes+1
                          dataset_file='',
                          device=device,
                          lr_backbone=1e-5, # Learning rate of the backbone
                          enc_layers=6, # Number of encoder layers
                          dec_layers=6, # Number of decoder layers
                          pre_norm=False,
                          set_cost_class=1, # Class coefficient in the matching cost
                          set_cost_bbox=5, # L1 box coefficient in the matching cost
                          set_cost_giou=2, # giou box coefficient in the matching cost
                          bbox_loss_coef=5,
                          giou_loss_coef=2,
                          eos_coef=0.1, # Relative classification weight of the no-object class
                          hidden_dim=256,
                          nheads=8,
                          num_encoder_layers=6,
                          num_decoder_layers=6,
                          dim_feedforward=2048,
                          dropout=0.1,
                          backbone='resnet50',
                          dilation=False,
                          position_embedding='sine',
                          num_queries=100,
                          aux_loss=False,
                          masks=False
                    )

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, criterion, postprocessors = build_model(args)
model = model.to(device)
print(f"Model built successfully on {device}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

/home/hice1/wzhu97/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hice1/wzhu97/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model built successfully on cuda
Number of parameters: 41,501,895


# 3. Verify the forward pass works

In [6]:
# Load data use the code I tested before
from utilities.dataset import create_dataloaders

coco_fn = '/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23/Annotations/instances_merged.json'
fn_frame_list = '/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23_frame_list.subset_0.01.csv' 

# Split the data into train, test, validation
# Perform transformations: resizing, normalization
# num_workers: Enables multi-process data loading 
train_loader, val_loader, test_loader = create_dataloaders(frames_csv=fn_frame_list,
                                                           annotation_path=coco_fn, 
                                                           test_size=0.2, val_size=0.1,
                                                           batch_size=2, num_workers=4,
                                                           random_state=42)



# Frames with >1 annotations were excluded: N=32

# ##### Data split summary #####
# Split      Videos   Frames
# Train          60      195
# Val            18       57
# Test            9       27
# Total          87      279


In [7]:
images, annotations = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    outputs = model(images)

print(f"Input shape:         {images.shape}")                  # [2, 3, 720, 1280]
print(f"Pred logits shape:   {outputs['pred_logits'].shape}")  # [2, 100, 2]
print(f"Pred boxes shape:    {outputs['pred_boxes'].shape}")   # [2, 100, 4]

Input shape:         torch.Size([2, 3, 720, 1280])
Pred logits shape:   torch.Size([2, 100, 3])
Pred boxes shape:    torch.Size([2, 100, 4])


# 4. Set up the training loop

In [1]:
from utilities.coco_utils import coco_annos_to_detr_targets

# 1. check output shapes are as expected
print(f"Pred logits: {outputs['pred_logits'].shape}")   # should be [2, 100, 2]
print(f"Pred boxes:  {outputs['pred_boxes'].shape}")    # should be [2, 100, 4]

# 2. check criterion works with your annotations
loss_dict = criterion

(outputs, annotations)
print(f"Losses: {loss_dict}")

In [18]:
from utilities.coco_utils import coco_annos_to_detr_targets

images, annotations = next(iter(train_loader))
images  = images.to(device)

# The original annotations do not match DETR's expectation
# Need to convert to {'labels':[], 'boxes':[]}
targets = coco_annos_to_detr_targets(annotations, img_h=720, img_w=1280, device=device)

# verify format
print(f"# Num targets:          {len(targets)}")           # 2 (one per image)
print(f"# Target 0 labels:      {targets[0]['labels']}")   # tensor([0]) or tensor([])
print(f"# Target 0 boxes:       {targets[0]['boxes']}")    # tensor([[cx, cy, w, h]])

# now criterion works
with torch.no_grad():
    outputs   = model(images)
    loss_dict = criterion(outputs, targets)
print(f"# Losses: {loss_dict}")

# Num targets:          2
# Target 0 labels:      tensor([0], device='cuda:0')
# Target 0 boxes:       tensor([[0.7394, 0.4067, 0.0665, 0.0827]], device='cuda:0')
# Losses: {'loss_ce': tensor(1.6015, device='cuda:0'), 'class_error': tensor(0., device='cuda:0'), 'loss_bbox': tensor(1.1453, device='cuda:0'), 'loss_giou': tensor(0.9985, device='cuda:0'), 'cardinality_error': tensor(99., device='cuda:0')}
